In [ ]:
library(igraph)
library(bipartite)
source("Functions.R") #include some helpers

## 2B Global scale (community)
Community structure and mixing patterns (community-scale)

## Connected components

In the simple network we saw at the start of the previous notebook we saw that for **every** pair of nodes, we can find a path connecting them. This is the definition of a **connected graph**. We can check this property for a given graph:

In [ ]:

g <- make_empty_graph(directed=FALSE) +
     vertices(1:5) +
     edges(c(1,2, 2,3, 3,1, 4,5))
plot(g)
is_connected(g)

In graph theory, a **connected component** (or just component) of an undirected graph is a subgraph in which any two vertices are connected to each other by paths.  
While visually, we can identify two connected components in that graph, it is not always so simple, and we need a way to verify this. 
NetworkX provides a function to let us find all of the connected components:`components(g)$no`.


In [ ]:
igraph::components(g)$no

The `components(g)` function takes a graph and returns:

- `comp$no`          # number of components
- `comp$csize`       # size of each component
- `comp$membership`  # component ID of each vertex

In [ ]:
comp<-igraph::components(g)
comp

We often care about the **largest connected component**, which is sometimes referred to as the **core** of the network. We search for the compatment with larger size, and take the name of that compartment, and get the subgraph with only those nodes.

In [ ]:
giant_id <- which.max(comp$csize)
g_giant <- igraph::induced_subgraph(g, comp$membership == giant_id)
plot(g_giant)



### Directed components
Directed networks have two kinds of connectivity. **Strongly connected** means that there exists a directed path between every pair of nodes, i.e., that from any node we can get to any other node while following edge directionality. Think of cars on a network of one-way streets: they can't drive against the flow of traffic.

In [ ]:
D <- graph_from_edgelist(
  matrix(c(
    1,2,
    2,3,
    3,2, 3,4, 3,5,
    4,2, 4,5, 4,6,
    5,6,
    6,4
  ), ncol = 2, byrow = TRUE),
  directed = TRUE
)

plot(D, vertex.label = V(D)$name)

In [ ]:
igraph::is_connected(D, mode="strong")

**Weakly connected** means that there exist a path between every pair of nodes, regardless of direction. Think about pedestrians on a network of one-way streets: they walk on the sidewalks so they don't care about the direction of traffic.

In [ ]:
igraph::is_connected(D, mode="weak")

> Note 1: If a network is strongly connected, it is also weakly connected. The converse is not always true, as seen in this example.


In the directed case, instead of `components(g)` we now have `components(D, mode = "weak")` and `components(D, mode = "strong")`:

In [ ]:
comp_w <- igraph::components(D, mode = "weak")
comp_w$no        # number of weak components
comp_s <- igraph::components(D, mode = "strong")
comp_s$no        # number of weak components

## Partitions

A **partition** of a graph is a separation of its nodes into disjoint groups. Consider the following graph:

In [ ]:
G <- make_empty_graph(directed = FALSE) |>
  add_vertices(8) |>
  add_edges(c(
    1,2, 2,3, 3,4, 4,1,   # cycle (0,1,2,3) -> (1,2,3,4)
    5,6, 6,7, 7,8, 8,5,   # cycle (4,5,6,7) -> (5,6,7,8)
    1,8                   # edge (0,7) -> (1,8)
  ))

plot(G, vertex.label = V(G)$name)

There are two **trivial partitions** in all networks:

1. The partition with one set containing every node;
2. The partition with N sets, each containing a single node.

A valid partition thus contains between 1 and N sets.

In [ ]:
#for example
membership1 <- c(1,1,1,1,  2,2,2,2)  # length = vcount(g)

In [ ]:
cols <- membership_to_colors(membership1)

plot(G,
     vertex.color = cols,
     vertex.label = V(G)$name)

In [ ]:
membership2 <- c(1,2,2,2,  3,3,3,1)  # length = vcount(g)
cols <- membership_to_colors(membership2)
plot(G,
     vertex.color = cols,
     vertex.label = V(G)$name)

## Modularity

At a high level, network community detection consists of finding a partition that achieves good separation between the groups of nodes. Before we get into how to find good partitions of a graph, we need an objective -- a way to measure how good the partition is. Modularity is one such objective function.

The modularity of a graph partition compares the number of intra-group edges with a random baseline. Higher modularity scores correspond to a higher proportion of intra-group edges, therefore fewer inter-group edges and better separation of groups.

For weighted undirected networks, we have
\begin{equation}
    Q_w=\frac{1}{W}\sum_C \left(W_C-\frac{s_C^2}{4W}\right),
\end{equation}
where 
* $W$ is the total weight of the links of the network,
* $W_C$ the total weight of the internal links of cluster $C$, and
* $s_C$ the total strength of the nodes of $C$.

The total weight $W$ is half the total strength for the same reason that the number of edges $L$ is half the total degree. While this formula may look a bit complicated, it's straightforward to write code to compute the sum, but still *igraph* and *bipartite* provides a modularity function that is more efficient than one I can write: `igraph::modularity`

In [ ]:
Q1  <- igraph::modularity(G, membership1)
Q2  <- igraph::modularity(G, membership2)
Q1
Q2

The trick is to find the partition that MAXIMIZES the modularity, and for that we use differnt algorithms that are already implemented.

In [ ]:
Q_louvain <- igraph::modularity(cluster_louvain(G))
Q_infomap  <- igraph::modularity(cluster_infomap(G))
Q_louvain
Q_infomap

<div style="background-color:#d4edda; border-left:6px solid #28a745; padding:12px; border-radius:4px; color:#000;"><b> Up to you:</b>

<h3> Exercise 10 </h3>
Compute the modularity of the foodweb of St Marks estuary using these two algorithms and compare the values
</div>

In [ ]:
#continue your code here
FW_filename<- "./Data/FW_st_marks.csv"

In [ ]:
# SOLUTION: uncomment line below to load solution
#load_and_show("./snippets/ex10.R")

In [ ]:
cluster_infomap(FW)

For bipartite networks we use the *bipartite* package.

Here modularity takes into account that there are two different guilds.

In [ ]:
P_Filename<-"./Data/Medan_Rio_Blanco.csv"
df <- read.csv(P_Filename, row.names = 1)
I <- as.matrix(df)
mod <- bipartite::computeModules(I)   # finds bipartite modules
plotModuleWeb(mod)   

In [ ]:
Qb <- slot(mod, "likelihood") #store modularity
Qb
#we can aslo retrieve the compartment of each species
listModuleInformation(mod)

<div style="background-color:#d4edda; border-left:6px solid #28a745; padding:12px; border-radius:4px; color:#000;"><b> Up to you:</b>

<h3> Exercise 11 </h3>
Now that you know how to compute the modularity of one graph, compute the modularity in 10 randomizations of the graph. 
Consider both the null model that preservers row/col sums as the simple null model we saw before that just keeps L.

Represent the ditribution of the values in the random model, and the value of the empirical network.
</div>

> Hint: you can generate a null model that keeps fixed L with `G0 <- igraph::sample_bipartite(Na, Np, m = L, type="gnm",directed = FALSE)` (keep in mind it is a graph, not a matrix, and the null model that keeps the row/col sums with `I0<-bipartite::nullmodel(I, N = 1, method = "r2dtable")[[1]]` (this is a matrix)

In [ ]:
#your code here

In [ ]:
# SOLUTION: uncomment line below to load solution 
#first null model
#load_and_show("./snippets/ex11A.R")

In [ ]:
#load_and_show("./snippets/ex11B.R") #plot of the first null model


In [ ]:
#load_and_show("./snippets/ex11C.R") #second null model

In [ ]:
#load_and_show("./snippets/ex11D.R") #plpot of second null model


## Nestedness

The level of nestedness of the mutualistic matrix is usually estimated by means of appropriate software. However, Bastolla et al.  introduced an explicit definition of nestedness that makes the calculation more straightforward and had the advantage of being related to the form of the matrix of interactions. 

$$
\eta_{B}=\frac{\sum_{i<j}\hat{n}_{ij}}{\sum_{i<j}\min(k_{i},k_{j})},
$$
Here min(, ) refers to the smaller of the two values and , and the number of shared symbiotic partners,
$\hat{n}_{ij}=\sum_{l}\hat{a}_{il}\hat{a}_{lj}=(\hat{a}^{2})_{ij}$
This nestedness index ranges from zero to one, and is highly correlated with previous measures of nestedness.
These analytic version is however not implemented in bipartite, so we will focuss on NODF, whih is the most popular.

One can compute the nestedness of a bipartite network like this:



In [ ]:
NODF<-bipartite::nested(I, method = "NODF")
visweb(I, type = "nested",labsize = 4)
as.numeric(NODF)

<div style="background-color:#d4edda; border-left:6px solid #28a745; padding:12px; border-radius:4px; color:#000;"><b> Up to you:</b>

<h3> Exercise 12 </h3>
Do the same thing for nestedness. 

Finally, plot the scatterplot of modularity vs nestedness in both null models. Plot also the empirical point.

Compute the correlation between modularity and nestedness.
</div>

> Note: Because we intend to quantify how nestedness and modularity are related in different networks we need to quantify both nestedness and modularity in the SAME network

In [ ]:
# SOLUTION: uncomment line below to load solution ; Compute the first null model
#load_and_show("./snippets/ex12A.R")

In [ ]:
# SOLUTION: uncomment line below to load solution ; Compute the second null model
#load_and_show("./snippets/ex12B.R")

In [ ]:
# SOLUTION: uncomment line below to load solution ; plot everything
#load_and_show("./snippets/ex12C.R") 